# POSM MOT coil — simulation of the **actual routed KiCad PCB**

**Source:** `POSM_MOT_Coil_60mmOD_35mmID_7Active_8Stack.kicad_pcb` (included alongside this notebook). Two identical boards, 60 mm OD / 35 mm aperture, **7 active winding layers on an 8-copper-layer stack**. Each winding has **5.15 geometric revolutions**, or **36.05 revolutions summed across seven layers per board**. That is not 36 concentric turns on a single layer.

This notebook models the **actual straight copper segments from the PCB file**, not idealized circular coils. It includes (1) axial B-field and gradient, (2) 2D field lines, (5) 3D field visualization, (6) resistance/current/voltage/power and spacing/current sweep. Numbering is a practical interpretation of the requested options 1,2,5,6; prior option definitions were not provided.

**Important:** Opposite winding polarity between the two PCB copies is assumed for anti-Helmholtz; the exact physical mounting and electrical wiring must be checked. Board spacing, copper thickness, via plating, operating temperature and trace continuity are **not confirmed**. Numerical results are predictions, not hardware validation. No KiCad DRC is performed here.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, re, pathlib
from collections import Counter
PCB = pathlib.Path('POSM_MOT_Coil_60mmOD_35mmID_7Active_8Stack.kicad_pcb')
assert PCB.exists(), f'Put {PCB.name} beside this notebook'
text = PCB.read_text()
# User-adjustable assumptions; distances in millimeters.
SEPARATION_MM = 10.0   # board midplane to board midplane: change after Mahdi confirms
COPPER_OZ = 1.0         # assumed finished copper weight per active layer
TRACE_WIDTH_MM = 0.30
VIA_PLATING_UM = 20.0   # rough assumption, not manufacturer specification
VIA_DRILL_MM = 0.25
TEMP_C = 20.0
TARGET_G_PER_CM = 10.0  # minimum of requested 10–20 G/cm
MAX_CURRENT_A = 1.0     # exploratory current cap; thermal limit NOT established
# 8-layer stack: nominal 1.6-mm PCB, approximate copper plane positions.
# Update with actual fabricator's layer stackup before reporting precise numbers.
LAYER_Z_MM = {'F.Cu':0.75,'In1.Cu':0.53,'In2.Cu':0.31,'In3.Cu':0.09,
              'In4.Cu':-0.13,'In5.Cu':-0.35,'In6.Cu':-0.57,'B.Cu':-0.75}
MU0=4*np.pi*1e-7
print('PCB:', PCB.name, '| separation:', SEPARATION_MM, 'mm')

In [ ]:
# Parse each physically routed copper segment, preserving layer and direction.
pattern = re.compile(r'\(segment \(start ([-\d.]+) ([-\d.]+)\) \(end ([-\d.]+) ([-\d.]+)\) \(width ([-\d.]+)\) \(layer "([^"]+)"\)')
raw = [(float(a),float(b),float(c),float(d),float(w),lay) for a,b,c,d,w,lay in pattern.findall(text)]
assert len(raw)>1000, f'Unexpected segment count {len(raw)}'
print('Copper segments:',len(raw),'by layer:',dict(Counter(r[-1] for r in raw)))
# The segments in the file follow the routed copper order within each layer.
# For Biot-Savart, assign a direction consistent with all active layers having
# the SAME rotational current sense; reverse the sequence where necessary.
def layer_wires(layer):
    segs=[r for r in raw if r[-1]==layer]
    start=np.array([[r[0],r[1]] for r in segs]); end=np.array([[r[2],r[3]] for r in segs])
    cross=(start[:,0]*end[:,1]-start[:,1]*end[:,0]).sum()
    if cross<0: start,end=end,start
    z=LAYER_Z_MM[layer]
    p=np.column_stack([start,np.full(len(start),z)])*1e-3
    q=np.column_stack([end,np.full(len(end),z)])*1e-3
    return p,q
active=['F.Cu','In1.Cu','In2.Cu','In3.Cu','In4.Cu','In5.Cu','In6.Cu']
P,Q=map(np.vstack,zip(*(layer_wires(lay) for lay in active)))
# Additional B.Cu leadouts and interlayer vias are excluded from field model.
# Their contributions to field are usually small but nonzero; this is an approximation.
print('Winding segments:',len(P),'| summed winding length (m):',round(np.linalg.norm(Q-P,axis=1).sum(),3))

In [ ]:
# Biot–Savart field using exact finite straight-segment expression.
# Numerical Gauss conversion: 1 T = 10,000 G.
def wire_field(points_m, p, q, chunk=150):
    pts=np.atleast_2d(points_m).astype(float)
    out=np.zeros((len(pts),3))
    dl=q-p; L=np.linalg.norm(dl,axis=1); u=dl/L[:,None]
    for k in range(0,len(pts),chunk):
        r=pts[k:k+chunk,None,:]-p[None,:,:]
        a=np.einsum('ijk,jk->ij',r,u)
        perp=r-a[:,:,None]*u[None,:,:]
        rho2=np.einsum('ijk,ijk->ij',perp,perp)
        # Avoid evaluating on a filament; copper has finite width in reality.
        rho2=np.maximum(rho2, (0.15e-3)**2)
        factor=(a/np.sqrt(rho2+a*a)-(a-L[None,:])/np.sqrt(rho2+(a-L[None,:])**2))/rho2
        out[k:k+chunk] = MU0/(4*np.pi)*np.einsum('ijk,ij->ik',np.cross(u[None,:,:],perp),factor)
    return out

def pair_field(points_mm, current_A=1.0, separation_mm=SEPARATION_MM):
    xyz=np.atleast_2d(points_mm)*1e-3
    shift=np.array([0.,0.,separation_mm*0.5e-3])
    # Same board geometry, opposite current signs in a COMMON global frame.
    return current_A*(wire_field(xyz,P+shift,Q+shift)-wire_field(xyz,P-shift,Q-shift))*1e4

def grad_at_center(separation_mm=SEPARATION_MM, h_mm=0.2):
    b=pair_field(np.array([[0,0,-h_mm],[0,0,h_mm]]),1.0,separation_mm)[:,2]
    # G/mm -> G/cm
    return (b[1]-b[0])/(2*h_mm)*10

grad_1A=grad_at_center()
I_min=TARGET_G_PER_CM/abs(grad_1A) if abs(grad_1A)>1e-10 else np.inf
print(f'Center axial gradient at 1 A: {grad_1A:+.3f} G/cm (magnitude {abs(grad_1A):.3f})')
print(f'Current for {TARGET_G_PER_CM:g} G/cm: {I_min:.3f} A')
print('Below exploratory current cap?',I_min<=MAX_CURRENT_A)
print('NOTE: current cap is NOT a verified safe thermal rating.')

In [ ]:
# OPTION 1: axial field, field zero and gradient around center
z=np.linspace(-20,20,201)
B=pair_field(np.column_stack([np.zeros_like(z),np.zeros_like(z),z]),I_min if np.isfinite(I_min) else 1.)
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].plot(z,B[:,2]);axs[0].axhline(0,color='gray',lw=.7);axs[0].set(xlabel='z (mm)',ylabel='Bz (G)',title='Axial magnetic field')
axs[1].plot(z,np.gradient(B[:,2],z)*10);axs[1].set(xlabel='z (mm)',ylabel='dBz/dz (G/cm)',title='Axial gradient')
for a in axs:a.grid(True,alpha=.3)
plt.tight_layout();plt.show()

In [ ]:
# OPTION 2: field-line plot in x–z midplane
x=np.linspace(-15,15,51); z2=np.linspace(-15,15,51)
X,Z=np.meshgrid(x,z2); pts=np.column_stack([X.ravel(),np.zeros(X.size),Z.ravel()])
field=pair_field(pts,I_min if np.isfinite(I_min) else 1.).reshape(*X.shape,3)
Bx,Bz=field[:,:,0],field[:,:,2]
fig,ax=plt.subplots(figsize=(7,7))
ax.streamplot(x,z2,Bx,Bz,density=1.4,color=np.hypot(Bx,Bz),cmap='viridis',linewidth=1)
ax.axhline(SEPARATION_MM/2,color='tab:red',ls='--',label='PCB planes')
ax.axhline(-SEPARATION_MM/2,color='tab:red',ls='--')
ax.set(xlabel='x (mm)',ylabel='z (mm)',title='Anti-Helmholtz field lines (y=0)',aspect='equal')
ax.legend();plt.show()

In [ ]:
# OPTION 5: 3D field vectors, plus representative spiral geometry
from mpl_toolkits.mplot3d import Axes3D
v=np.linspace(-10,10,7);xx,yy,zz=np.meshgrid(v,v,v,indexing='ij')
xyz=np.column_stack([xx.ravel(),yy.ravel(),zz.ravel()])
B3=pair_field(xyz,I_min if np.isfinite(I_min) else 1.)
norm=np.linalg.norm(B3,axis=1); unit=B3/np.maximum(norm[:,None],1e-12)
fig=plt.figure(figsize=(10,8));ax=fig.add_subplot(111,projection='3d')
ax.quiver(xyz[:,0],xyz[:,1],xyz[:,2],unit[:,0],unit[:,1],unit[:,2],length=1.7,normalize=False,alpha=.6)
# Draw F.Cu traces on both boards, sparse for plotting speed.
p0,q0=layer_wires('F.Cu')
for sign in (-1,1):
    zshift=sign*SEPARATION_MM/2
    for j in range(0,len(p0),3):
        ax.plot([p0[j,0]*1e3,q0[j,0]*1e3],[p0[j,1]*1e3,q0[j,1]*1e3],
                [p0[j,2]*1e3+zshift,q0[j,2]*1e3+zshift],color='saddlebrown',lw=.7)
ax.set(xlabel='x (mm)',ylabel='y (mm)',zlabel='z (mm)',title='3D field directions + top winding on both PCBs')
ax.set_box_aspect([1,1,.8]);plt.show()

In [ ]:
# OPTION 6: winding resistance, voltage, power, and spacing/current sweep
# Copper: 1 oz/ft² ~ 34.8 µm; resistivity at 20 C ~ 1.724e-8 ohm*m.
rho20=1.724e-8; alpha=0.00393
rho=rho20*(1+alpha*(TEMP_C-20))
thick=COPPER_OZ*34.8e-6
Lw=np.linalg.norm(Q-P,axis=1).sum()
R_winding=rho*Lw/(TRACE_WIDTH_MM*1e-3*thick)
# Leadout lengths (B.Cu) extracted separately; estimate from the routed segments.
lead=[r for r in raw if r[-1]=='B.Cu']
Llead=sum(np.hypot(r[2]-r[0],r[3]-r[1]) for r in lead)*1e-3
Rlead=rho*Llead/(TRACE_WIDTH_MM*1e-3*thick)
# 8 plated through vias total, estimate cylindrical plating cross-section.
via_count=len(re.findall(r'^\(via ',text,re.M))
via_len=1.6e-3
via_area=np.pi*(VIA_DRILL_MM*1e-3)*(VIA_PLATING_UM*1e-6)
Rvia=via_count*rho*via_len/via_area
Rboard=R_winding+Rlead+Rvia
# Two identical PCBs in SERIES -> equal current and resistance doubled.
Rpair=2*Rboard
I=I_min
print(f'Active winding copper length / board: {Lw:.3f} m')
print(f'B.Cu leadout length / board: {Llead:.3f} m; via count: {via_count}')
print(f'R winding: {R_winding:.3f} Ω; leadouts: {Rlead:.3f} Ω; estimated vias: {Rvia:.3f} Ω')
print(f'Estimated R / PCB: {Rboard:.3f} Ω; two PCBs series: {Rpair:.3f} Ω')
if np.isfinite(I):
    print(f'For {TARGET_G_PER_CM:g} G/cm: I={I:.3f} A, V_pair={I*Rpair:.3f} V, P_pair={I*I*Rpair:.3f} W')
    print(f'Each PCB: V={I*Rboard:.3f} V, P={I*I*Rboard:.3f} W')
print('R and power exclude contact/driver losses; copper heating changes resistance.')
spacings=np.arange(4,21,2)
grads=np.array([abs(grad_at_center(float(s))) for s in spacings]); required=TARGET_G_PER_CM/np.maximum(grads,1e-12)
print('Spacing (mm) | Gradient at 1A (G/cm) | Current for 10 G/cm (A)')
for s,g,cur in zip(spacings,grads,required):print(f'{s:>12.0f} | {g:>22.3f} | {cur:>24.3f}')
fig,ax=plt.subplots(figsize=(7,4));ax.plot(spacings,required,'o-');ax.axhline(MAX_CURRENT_A,color='r',ls='--',label='Exploratory cap, not thermal limit')
ax.set(xlabel='PCB center-to-center separation (mm)',ylabel='Required current (A)',title='Current for 10 G/cm versus spacing');ax.grid(alpha=.3);ax.legend();plt.show()

## Engineering checks before manufacturing or applying current

- **Verify physical continuity and DRC in KiCad**: the field model assumes all seven winding layers form the intended series conductor. A single-net label does not prove continuity or exclude shorts.
- **Confirm the mechanical separation and orientation**: the simulation assumes two coaxial identical boards at `±SEPARATION_MM/2` and opposite currents viewed in one fixed coordinate system.
- **Confirm copper thickness and actual stackup**: the notebook uses assumed copper layer z coordinates, 1 oz copper and 20 µm via plating. Replace these with fab values.
- **Confirm thermal performance**: estimated Joule heating is not a temperature-rise or current-carrying-capacity validation. Consider the narrow 0.3 mm traces and via bottlenecks.
- **Review gradient target**: computed `dBz/dz` is axial gradient. In a current-free symmetric quadrupole region, transverse gradients are approximately half the axial magnitude. Confirm which axis Mahdi means by 10–20 G/cm.
- **No simulation result can make a PCB manufacturing-ready** without electrical and mechanical review.